# COMP 579 Assignment 3 - Question 1

Prepared by: Nicolas Smits

Presented to: Prof. Isabeau Premont Schwartz, Valliappan Chidambaram Adaikkappan

## Task 1 - Value Based Methods with Deep Neural Networks

In [17]:
import gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import os
import random
from tqdm import tqdm

if not hasattr(np, 'float_'):
    np.float_ = np.float64

if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

### 1.1 - Q-Learning and Expected SARSA with Deep NN Function Approximation

In [21]:
# Q-network Function Approximator
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=None):
        super(QNetwork, self).__init__()
        if hidden_size is None:
            hidden_size = [256, 256]
        layers = []
        last_dim = state_size # input layer init
        # populate hidden layers
        for hidden_dim in hidden_size:
            layers.append(nn.Linear(last_dim, hidden_dim))
            layers.append(nn.ReLU())
            last_dim = hidden_dim
        # output layer
        layers.append(nn.Linear(last_dim, action_size))
        self.model = nn.Sequential(*layers)
        self.apply(init_weights) # initialize weights (method in cell below)

    def forward(self, x):
        return self.model(x)
    
# Replay Buffer
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    # store experience in replay buffer
    def push(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity: # buffer not full
            self.buffer.append(None) 
        self.buffer[self.position] = (state, action, reward, next_state, done) # store 
        self.position = (self.position + 1) % self.capacity # update position

    # sample from buffer
    def sample(self, batch_size):
      batch = random.sample(self.buffer, batch_size)
      states, actions, rewards, next_states, dones = zip(*batch)
      return (np.stack(states), 
              np.array(actions),
              np.array(rewards, dtype=np.float32),
              np.stack(next_states),
              np.array(dones, dtype=np.uint8))
    
    def __len__(self):
        return len(self.buffer)

# Agent Superclass
class Agent:
    def __init__(self, env, alpha=1e-3, gamma=0.99, epsilon=0.1):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

        # assume the obseration space is a vector
        state_space = env.observation_space.shape[0] # state space dimension
        action_space = env.action_space.n # action space dimension

        # for GPU acceleration
        if torch.backends.mps.is_available():
            self.device = torch.device("mps") # m1 macos gang wya
        elif torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")

        print(f"Using device: {self.device}")

        # Q-network
        self.q_network = QNetwork(state_space, action_space).to(self.device) # initialize Q-network
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=alpha) # optimizer
        self.loss_fn = nn.MSELoss() # loss function
        # self.loss_fn = nn.SmoothL1Loss() # alternative
    
    # epsilon-greedy policy
    def select_action(self, state, greedy=False):
        if greedy:
            return self.q_network(torch.tensor(state, dtype=torch.float32).to(self.device)).argmax().item()
        else:
            if np.random.rand() < self.epsilon:
                return self.env.action_space.sample() # random action
            else:
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device) # convert state to tensor
                with torch.no_grad(): # no gradient calculation
                        q_values = self.q_network(state_tensor) # greedy action
                return int(torch.argmax(q_values, dim=1).item())
            
    # update Q-network (general method, plug in target computation from agent)
    def update(self, s, a, r, s_, done):
        """
        Update Q-network.

        Args:
            s: current state
            a: current action
            r: reward
            s_: next state
            done: whether the episode is done
        Returns:
            loss: loss value
        """
        self.optimizer.zero_grad()

        state_tensor = torch.FloatTensor(s).unsqueeze(0).to(self.device)
        next_state_tensor = torch.FloatTensor(s_).unsqueeze(0).to(self.device)

        q_values = self.q_network(state_tensor)
        predicted = q_values[0, a]

        # The algorithm-specific target computation is done in compute_target.
        target = self.compute_target(r, next_state_tensor, done) ## ALGORITHM-DEPENDENT (subclasses)
        loss = self.loss_fn(predicted, target)
        loss.backward()
        self.optimizer.step()
        return loss.item()
    
    # update Q-network for a batch of transitions
    def update_batch(self, batch):
        """
        Update Q-network for a batch of transitions.

        Args:
            batch: tuple of (states, actions, rewards, next_states, dones)

        Returns:
            loss: loss value
        """
        states, actions, rewards, next_states, dones = batch

        states_tensor = torch.FloatTensor(states).to(self.device)
        actions_tensor = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        rewards_tensor = torch.FloatTensor(rewards).to(self.device)
        next_states_tensor = torch.FloatTensor(next_states).to(self.device)
        dones_tensor = torch.FloatTensor(dones).to(self.device)

        q_values = self.q_network(states_tensor)
        # gather Q-values corresponding to taken actions.
        predicted = q_values.gather(1, actions_tensor).squeeze(1)
        
        target = self.compute_target_batch(rewards_tensor, next_states_tensor, dones_tensor) ## ALGORITHM-DEPENDENT (subclasses)
        loss = self.loss_fn(predicted, target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()
    
    # methods are overridden by the subclass.
    def compute_target(self, reward, next_state_tensor, done):
        raise NotImplementedError("Override in subclass")

    def compute_target_batch(self, rewards_tensor, next_states_tensor, dones_tensor):
        raise NotImplementedError("Override in subclass")      

    
class QLearningAgent(Agent):
    def __init__(self, env, alpha=1e-3, gamma=0.99, epsilon=0.1):
        super(QLearningAgent, self).__init__(env, alpha, gamma, epsilon)

    # update Q-network via Q-learning
    # @overrides(Agent)
    def compute_target(self, reward, next_state_tensor, done):
        if done:
            return torch.tensor(reward, dtype=torch.float32, device=self.device)
        else:
            with torch.no_grad():
                next_q_values = self.q_network(next_state_tensor)
            # Q-learning uses the maximum over next-state Q-values.
            max_next_q = torch.max(next_q_values)
            target = torch.tensor(reward, dtype=torch.float32, device=self.device) + self.gamma * max_next_q # Q target
            return target
    
    # @overrides(Agent)
    def compute_target_batch(self, rewards_tensor, next_states_tensor, dones_tensor):
        with torch.no_grad():
            next_q_values = self.q_network(next_states_tensor)
            max_next_q_values, _ = next_q_values.max(dim=1)
        target = rewards_tensor + self.gamma * max_next_q_values * (1 - dones_tensor)
        return target

class ExpectedSarsaAgent(Agent):
    def __init__(self, env, alpha=0.001, gamma=0.99, epsilon=0.1):
        super().__init__(env, alpha, gamma, epsilon)

    # @overrides(Agent)
    def compute_target(self, reward, next_state_tensor, done):
        if done:
            return torch.tensor(reward, dtype=torch.float32, device=self.device)
        else:
            with torch.no_grad():
                # compute Q-values for the next state from the Q-network
                next_q_values = self.q_network(next_state_tensor)

            num_actions = next_q_values.size(1) # num actions

            # compute epsilon-greedy probabilities.
            best_action = torch.argmax(next_q_values, dim=1).item() # best action
            probs = torch.ones(num_actions, device=self.device) * (self.epsilon / num_actions) # epsilon-greedy probs
            probs[best_action] += (1.0 - self.epsilon) # update best action prob
            expected_q = torch.sum(next_q_values[0] * probs) 
            target = torch.tensor(reward, dtype=torch.float32, device=self.device) + self.gamma * expected_q
        
            return target
        
    # @override(Agent)
    def compute_target_batch(self, rewards_tensor, next_states_tensor, dones_tensor):
        with torch.no_grad():
            # compute Q-values for the next state from the Q-network
            next_q_values = self.q_network(next_states_tensor)
            num_actions = next_q_values.size(1) # num actions
            best_actions = next_q_values.argmax(dim=1) # best actions

            # build a probability tensor for the epsilon-greedy policy.
            probs = torch.ones_like(next_q_values) * (self.epsilon / num_actions)
            probs[range(probs.shape[0]), best_actions] += (1.0 - self.epsilon)
            expected_q = (next_q_values * probs).sum(dim=1)
        target = rewards_tensor + self.gamma * expected_q * (1 - dones_tensor)

        return target

# Training
def train_agent(agent, env, num_episodes=1000):
    """
    Train the agent in the given environment.

    Args:
        agent: Agent object
        env: OpenAI Gym environment
        num_episodes: number of episodes to train the agent
    Returns:
        None
    """
    agent.total_rewards = [] # store total rewards for each episode
    for episode in range(num_episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0
        while not done:
            actions = env.action_space.n # get number of actions
            action = agent.select_action(state) # select action
            next_state, reward, terminated, truncated, info = env.step(action) # take action and observe next state and reward
            done = terminated or truncated
            total_reward += reward # update total reward
            loss = agent.update(state, action, reward, next_state, done) # update Q-network
            state = next_state
        agent.total_rewards.append(total_reward)
        print(f"Episode: {episode + 1}, Total Reward: {total_reward}")
    return agent.total_rewards

# Training agent with checkpointing
def train_agent_with_checkpoint(agent, env, num_episodes=1000, 
                                checkpoint_path="checkpoint.pt", checkpoint_interval=500):
    """
    Train the agent in the given environment with checkpointing using a tqdm progress bar.
    Saves:
      - Agent network state and optimizer state.
      - Total rewards.
    """
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=agent.device)
        starting_episode = checkpoint['episode'] + 1
        agent.q_network.load_state_dict(checkpoint['model_state_dict'])
        agent.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        total_rewards = checkpoint['total_rewards']
        tqdm.write(f"Resuming training from episode {starting_episode}")
    else:
        starting_episode = 0
        total_rewards = []
        tqdm.write("Starting new training run.")

    # Wrap the episode loop in a tqdm progress bar.
    progress_bar = tqdm(range(starting_episode, num_episodes), initial=starting_episode, total=num_episodes, desc="Episodes")
    
    for episode in progress_bar:
        state, _ = env.reset()
        done = False
        episode_reward = 0

        while not done:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            episode_reward += reward
            loss = agent.update(state, action, reward, next_state, done)
            state = next_state

        total_rewards.append(episode_reward)
        # Update the progress bar description with current episode stats.
        progress_bar.set_description(f"Ep {episode+1}: Reward = {episode_reward}")

        if (episode + 1) % checkpoint_interval == 0:
            checkpoint = {
                'episode': episode,
                'model_state_dict': agent.q_network.state_dict(),
                'optimizer_state_dict': agent.optimizer.state_dict(),
                'total_rewards': total_rewards
            }
            torch.save(checkpoint, checkpoint_path)
            tqdm.write(f"Checkpoint saved at episode {episode+1}")

    agent.total_rewards = total_rewards
    return total_rewards

# Training with replay buffer and a batch update
def train_agent_with_replay(agent, env, num_episodes=1000, batch_size=32, start_training=1000, update_every=4):
    """
    Train the agent in the given environment.

    Args:
        agent: Agent object
        env: OpenAI Gym environment
        num_episodes: number of episodes to train the agent
        batch_size: batch size for updating the Q-network"
        start_training: start training after this number of transitions
        update_every: update the Q-network every this number of steps
    Returns:
        None
    """

    replay_buffer = ReplayBuffer() # initialize replay buffer
    total_steps = 0
        
    agent.total_rewards = [] # store total rewards for each episode
    for episode in range(num_episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0
        total_loss = 0

        while not done:
            action = agent.select_action(state) # select action
            next_state, reward, terminated, truncated, info = env.step(action) # observe reward and next state
            done = terminated or truncated
            total_reward += reward
            
            # store transition in replay buffer
            replay_buffer.push(state, action, reward, next_state, done)
            state = next_state
            total_steps += 1

            # only update if we have enough transitions in the replay buffer
            if len(replay_buffer) > start_training and total_steps % update_every == 0:
                batch = replay_buffer.sample(batch_size) # sample from buffer
                loss = agent.update_batch(batch) # update based on batch
                total_loss += loss
            
        print(f"Episode: {episode + 1}, Total Reward: {total_reward}, Total Loss: {total_loss}")


# Training with replay buffer and a batch update with checkpointing
def train_agent_with_replay_checkpoint(agent, env, num_episodes=1000, batch_size=32,
                                       start_training=1000, update_every=4,
                                       checkpoint_path="replay_checkpoint.pt",
                                       checkpoint_interval=500):
    """
    Train the agent using a replay buffer with checkpointing.
    Saves:
      - Agent network state and optimizer state.
      - Total rewards and total steps.
      - Replay buffer contents.
    """
    replay_buffer = ReplayBuffer(capacity=int(1e6))
    total_steps = 0

    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=agent.device)
        starting_episode = checkpoint['episode'] + 1
        agent.q_network.load_state_dict(checkpoint['model_state_dict'])
        agent.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        total_rewards = checkpoint['total_rewards']
        total_steps = checkpoint['total_steps']
        replay_buffer.buffer = checkpoint['replay_buffer']
        replay_buffer.position = checkpoint['buffer_position']
        print(f"Resuming replay training from episode {starting_episode}")
    else:
        starting_episode = 0
        total_rewards = []
        print("Starting new replay training run.")

    for episode in range(starting_episode, num_episodes):
        state = env.reset()
        done = False
        episode_reward = 0
        total_loss = 0

        while not done:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            episode_reward += reward

            replay_buffer.push(state, action, reward, next_state, done)
            state = next_state
            total_steps += 1

            if len(replay_buffer) > start_training and total_steps % update_every == 0:
                batch = replay_buffer.sample(batch_size)
                loss = agent.update_batch(batch)
                total_loss += loss

        total_rewards.append(episode_reward)
        print(f"Episode {episode+1}: Reward = {episode_reward}, Loss = {total_loss}")

        # Save checkpoint every checkpoint_interval episodes.
        if (episode + 1) % checkpoint_interval == 0:
            checkpoint = {
                'episode': episode,
                'model_state_dict': agent.q_network.state_dict(),
                'optimizer_state_dict': agent.optimizer.state_dict(),
                'total_rewards': total_rewards,
                'total_steps': total_steps,
                'replay_buffer': replay_buffer.buffer,
                'buffer_position': replay_buffer.position
            }
            torch.save(checkpoint, checkpoint_path)
            print(f"Checkpoint saved at episode {episode+1}")

    agent.total_rewards = total_rewards
    return total_rewards
  
# to save results of different trials
def save_results(results, algorithm, environment, buffer_option, epsilon, alpha, trial):
    # Update folder path to point to Google Drive
    folder = f"/content/drive/MyDrive/COMP_579_A3/results/{environment}/{algorithm}_{buffer_option}"
    os.makedirs(folder, exist_ok=True)
    filename = f"{folder}/{algorithm}_{environment}_{buffer_option}_eps-{epsilon}_alpha-{alpha}_trial-{trial}.npy"
    np.save(filename, results)
    print(f"Saved results to {filename}")

### 1.2 - Model Configuration

In [22]:
# function to initialise Q-network weights
# called in the constructor of the QNetwork class (see above)
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.uniform_(m.weight, -0.001, 0.001)
        if m.bias is not None:
            nn.init.uniform_(m.bias, -0.001, 0.001)


### 1.3 - $\epsilon$-Greedy Policy for Different Initialisations

Acrobot v1 - Q-Learning

In [24]:
# epsilon greedy policy params
# epsilons = [0.2, 0.1, 0.05]
# step_sizes = [0.25, 0.125, 0.0625]

# epsilons = [0.1, 0.05]
# step_sizes = [0.25, 0.125, 0.0625]

epsilons = [0.1, 0.05]
step_sizes = [0.0001, 0.001, 0.01]

# run params
num_trials = 10
num_episodes = 1000

### Q-learning Acrobot-v1 ###

# initialise environment
env = gym.make("Acrobot-v1")
# env = gym.make("Acrobot-v1", new_step_API=True, render_mode="human")

env_name = "Acrobot-v1"
algorithm = "QLearning"
buffer_option = "NoReplay"

# initialise results
q_agent_acrobot_results = np.zeros((len(epsilons), len(step_sizes), num_trials, num_episodes))

# run experiments
for i, epsilon in enumerate(epsilons):
    for j, step_size in enumerate(step_sizes):
        print(f"Running Experiment for Epsilon = {epsilon}, Step Size = {step_size}")
        for trial in range(num_trials):
            agent = QLearningAgent(env, alpha=step_size, gamma=0.99, epsilon=epsilon)
            checkpoint_path = f"/results/{env_name}/{algorithm}_{buffer_option}/checkpoint_eps-{epsilon}_alpha-{step_size}_trial-{trial}.pt"
            rewards = train_agent_with_checkpoint(agent, env, num_episodes,
                                                  checkpoint_path=checkpoint_path,
                                                  checkpoint_interval=500)
            q_agent_acrobot_results[i, j, trial, :] = np.array(rewards)
            save_results(rewards, algorithm, env_name, buffer_option, epsilon, step_size, trial)
            print(f"Completed Trial {trial+1} for Epsilon = {epsilon}, Step Size = {step_size}")

# Save overall results for this configuration (optional)
np.save(f"/results/{env_name}/{algorithm}_{buffer_option}_overall.npy", q_agent_acrobot_results)


Running Experiment for Epsilon = 0.1, Step Size = 0.0001
Using device: mps
Starting new training run.


Ep 500: Reward = -185.0:  50%|████▉     | 499/1000 [18:29<18:34,  2.22s/it]


RuntimeError: Parent directory /results/Acrobot-v1/QLearning_NoReplay does not exist.

Acrobot v1 - Expected SARSA

In [ ]:
### Expected Sarsa Acrobot-v1 ###
num_trials = 10

# initialise environment
env = gym.make("Acrobot-v1")

env_name = "Acrobot-v1"
algorithm = "ExpectedSarsa"
buffer_option = "NoReplay"

# initialise results
expected_sarsa_agent_acrobot_results = np.zeros((len(epsilons), len(step_sizes), num_trials, num_episodes))

# run experiments
for i, epsilon in enumerate(epsilons):
    for j, step_size in enumerate(step_sizes):
        print(f"Running Experiment for Epsilon = {epsilon}, Step Size = {step_size}")
        for trial in range(num_trials):
            agent = ExpectedSarsaAgent(env, alpha=step_size, gamma=0.99, epsilon=epsilon)
            checkpoint_path = f"results/{env_name}/{algorithm}_{buffer_option}/checkpoint_eps-{epsilon}_alpha-{step_size}_trial-{trial}.pt"
            rewards = train_agent_with_checkpoint(agent, env, num_episodes,
                                                  checkpoint_path=checkpoint_path,
                                                  checkpoint_interval=50)
            expected_sarsa_agent_acrobot_results[i, j, trial, :] = np.array(rewards)
            save_results(rewards, algorithm, env_name, buffer_option, epsilon, step_size, trial)
            print(f"Completed Trial {trial+1} for Epsilon = {epsilon}, Step Size = {step_size}")

# Save overall results for this configuration (optional)
np.save(f"results/{env_name}/{algorithm}_{buffer_option}_overall.npy", expected_sarsa_agent_acrobot_results)


ALE/Assault-ram-v5 - Q-Learning

In [ ]:
# initialise environment
env = gym.make("ALE/Assault-ram-v5")

env_name = "Assault-ram-v5"
algorithm = "QLearning"
buffer_option = "NoReplay"

# initialise results
q_agent_ale_results = np.zeros((len(epsilons), len(step_sizes), num_trials, num_episodes))

# run experiments
for i, epsilon in enumerate(epsilons):
    for j, step_size in enumerate(step_sizes):
        print(f"Running Experiment for Epsilon = {epsilon}, Step Size = {step_size}")
        for trial in range(num_trials):
            agent = QLearningAgent(env, alpha=step_size, gamma=0.99, epsilon=epsilon)
            checkpoint_path = f"results/{env_name}/{algorithm}_{buffer_option}/checkpoint_eps-{epsilon}_alpha-{step_size}_trial-{trial}.pt"
            rewards = train_agent_with_checkpoint(agent, env, num_episodes,
                                                  checkpoint_path=checkpoint_path,
                                                  checkpoint_interval=50)
            q_agent_ale_results[i, j, trial, :] = np.array(rewards)
            save_results(rewards, algorithm, env_name, buffer_option, epsilon, step_size, trial)
            print(f"Completed Trial {trial+1} for Epsilon = {epsilon}, Step Size = {step_size}")

# Save overall results for this configuration (optional)
np.save(f"results/{env_name}/{algorithm}_{buffer_option}_overall.npy", q_agent_ale_results)


ALE/Assault-ram-v5 - Expected SARSA

In [ ]:
# initialise environment
env = gym.make("ALE/Assault-ram-v5")

env_name = "Assault-ram-v5"
algorithm = "ExpectedSarsa"
buffer_option = "NoReplay"

# initialise results
expected_sarsa_agent_ale_results = np.zeros((len(epsilons), len(step_sizes), num_trials, num_episodes))

# run experiments
for i, epsilon in enumerate(epsilons):
    for j, step_size in enumerate(step_sizes):
        print(f"Running Experiment for Epsilon = {epsilon}, Step Size = {step_size}")
        for trial in range(num_trials):
            agent = ExpectedSarsaAgent(env, alpha=step_size, gamma=0.99, epsilon=epsilon)
            checkpoint_path = f"results/{env_name}/{algorithm}_{buffer_option}/checkpoint_eps-{epsilon}_alpha-{step_size}_trial-{trial}.pt"
            rewards = train_agent_with_checkpoint(agent, env, num_episodes,
                                                  checkpoint_path=checkpoint_path,
                                                  checkpoint_interval=50)
            expected_sarsa_agent_ale_results[i, j, trial, :] = np.array(rewards)
            save_results(rewards, algorithm, env_name, buffer_option, epsilon, step_size, trial)
            print(f"Completed Trial {trial+1} for Epsilon = {epsilon}, Step Size = {step_size}")

# Save overall results for this configuration (optional)
np.save(f"results/{env_name}/{algorithm}_{buffer_option}_overall.npy", expected_sarsa_agent_ale_results)

### 1.4 - Repeat Using Replay Buffer

Acrobot-v1 - Q-Learning with Buffer

In [ ]:
# initialise environment
env = gym.make("Acrobot-v1")

env_name = "Acrobot-v1"
algorithm = "QLearning"
buffer_option = "Replay"

# initialise results
q_agent_acrobot_replay_results = np.zeros((len(epsilons), len(step_sizes), num_trials, num_episodes))

# run experiments
for i, epsilon in enumerate(epsilons):
    for j, step_size in enumerate(step_sizes):
        print(f"Running Replay Experiment for Epsilon = {epsilon}, Step Size = {step_size}")
        for trial in range(num_trials):
            agent = QLearningAgent(env, alpha=step_size, gamma=0.99, epsilon=epsilon)
            checkpoint_path = f"results/{env_name}/{algorithm}_{buffer_option}/checkpoint_eps-{epsilon}_alpha-{step_size}_trial-{trial}.pt"
            rewards = train_agent_with_replay_checkpoint(agent, env, num_episodes,
                                                         batch_size=32,
                                                         start_training=1000,
                                                         update_every=4,
                                                         checkpoint_path=checkpoint_path,
                                                         checkpoint_interval=50)
            q_agent_acrobot_replay_results[i, j, trial, :] = np.array(rewards)
            save_results(rewards, algorithm, env_name, buffer_option, epsilon, step_size, trial)
            print(f"Completed Replay Trial {trial+1} for Epsilon = {epsilon}, Step Size = {step_size}")

# Optionally, save overall results
np.save(f"results/{env_name}/{algorithm}_{buffer_option}_overall.npy", q_agent_acrobot_replay_results)


Acrobot-v1 - Expected SARSA with Buffer

In [ ]:
# initialise environment
env = gym.make("Acrobot-v1")

env_name = "Acrobot-v1"
algorithm = "ExpectedSarsa"
buffer_option = "Replay"

# initialise results
expected_sarsa_agent_acrobot_replay_results = np.zeros((len(epsilons), len(step_sizes), num_trials, num_episodes))

# run experiments
for i, epsilon in enumerate(epsilons):
    for j, step_size in enumerate(step_sizes):
        print(f"Running Replay Experiment for Epsilon = {epsilon}, Step Size = {step_size}")
        for trial in range(num_trials):
            agent = ExpectedSarsaAgent(env, alpha=step_size, gamma=0.99, epsilon=epsilon)
            checkpoint_path = f"results/{env_name}/{algorithm}_{buffer_option}/checkpoint_eps-{epsilon}_alpha-{step_size}_trial-{trial}.pt"
            rewards = train_agent_with_replay_checkpoint(agent, env, num_episodes,
                                                         batch_size=32,
                                                         start_training=1000,
                                                         update_every=4,
                                                         checkpoint_path=checkpoint_path,
                                                         checkpoint_interval=50)
            expected_sarsa_agent_acrobot_replay_results[i, j, trial, :] = np.array(rewards)
            save_results(rewards, algorithm, env_name, buffer_option, epsilon, step_size, trial)
            print(f"Completed Replay Trial {trial+1} for Epsilon = {epsilon}, Step Size = {step_size}")

# Optionally, save overall results
np.save(f"results/{env_name}/{algorithm}_{buffer_option}_overall.npy", expected_sarsa_agent_acrobot_replay_results)

ALE/Assault-ram-v5 - Q-Learning with Buffer

In [ ]:
# initialise environment
env = gym.make("ALE/Assault-ram-v5")

env_name = "Assault-ram-v5"
algorithm = "QLearning"
buffer_option = "Replay"

# initialise results
q_agent_ale_replay_results = np.zeros((len(epsilons), len(step_sizes), num_trials, num_episodes))

# run experiments
for i, epsilon in enumerate(epsilons):
    for j, step_size in enumerate(step_sizes):
        print(f"Running Replay Experiment for Epsilon = {epsilon}, Step Size = {step_size}")
        for trial in range(num_trials):
            agent = QLearningAgent(env, alpha=step_size, gamma=0.99, epsilon=epsilon)
            checkpoint_path = f"results/{env_name}/{algorithm}_{buffer_option}/checkpoint_eps-{epsilon}_alpha-{step_size}_trial-{trial}.pt"
            rewards = train_agent_with_replay_checkpoint(agent, env, num_episodes,
                                                         batch_size=32,
                                                         start_training=1000,
                                                         update_every=4,
                                                         checkpoint_path=checkpoint_path,
                                                         checkpoint_interval=50)
            q_agent_ale_replay_results[i, j, trial, :] = np.array(rewards)
            save_results(rewards, algorithm, env_name, buffer_option, epsilon, step_size, trial)
            print(f"Completed Replay Trial {trial+1} for Epsilon = {epsilon}, Step Size = {step_size}")

# Optionally, save overall results
np.save(f"results/{env_name}/{algorithm}_{buffer_option}_overall.npy", q_agent_ale_replay_results)

ALE/Assault-ram-v5 - Expected SARSA with Buffer

In [ ]:
# initialise environment
env = gym.make("ALE/Assault-ram-v5")

env_name = "Assault-ram-v5"
algorithm = "ExpectedSarsa"
buffer_option = "Replay"

# initialise results
expected_sarsa_agent_ale_replay_results = np.zeros((len(epsilons), len(step_sizes), num_trials, num_episodes))

# run experiments
for i, epsilon in enumerate(epsilons):
    for j, step_size in enumerate(step_sizes):
        print(f"Running Replay Experiment for Epsilon = {epsilon}, Step Size = {step_size}")
        for trial in range(num_trials):
            agent = ExpectedSarsaAgent(env, alpha=step_size, gamma=0.99, epsilon=epsilon)
            checkpoint_path = f"results/{env_name}/{algorithm}_{buffer_option}/checkpoint_eps-{epsilon}_alpha-{step_size}_trial-{trial}.pt"
            rewards = train_agent_with_replay_checkpoint(agent, env, num_episodes,
                                                         batch_size=32,
                                                         start_training=1000,
                                                         update_every=4,
                                                         checkpoint_path=checkpoint_path,
                                                         checkpoint_interval=50)
            expected_sarsa_agent_ale_replay_results[i, j, trial, :] = np.array(rewards)
            save_results(rewards, algorithm, env_name, buffer_option, epsilon, step_size, trial)
            print(f"Completed Replay Trial {trial+1} for Epsilon = {epsilon}, Step Size = {step_size}")

# Optionally, save overall results
np.save(f"results/{env_name}/{algorithm}_{buffer_option}_overall.npy", expected_sarsa_agent_ale_replay_results)

### 1.5 - Plot Results